# Classifying and Extracting from SEC Filings

<a href="https://colab.research.google.com/github/run-llama/llama_cloud_services/blob/main/examples/classify/sec_filing_classify_extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates how to classify and extract information from SEC filings using LlamaParse. We'll walk through the process of classifying a document as either a 10-K or 10-Q filing and then extracting the relevant information.

**Note**: The classification module is currently in *beta*, so we are still ironing out some interface/implementation details. Please let us know your feedback!

Status:
| Last Executed | Version | State      |
|---------------|---------|------------|
| Sep-09-2025   | 0.6.65  | Maintained |

## Overview

This notebook demonstrates a classify+extract workflow on SEC filings using LlamaCloud and LlamaIndex Workflows.

- Classify each document as one of: 10-K, 10-Q, 8-K, Proxy (DEF 14A)
- Extract a type-specific schema depending on the classification
- Orchestrate via an event-driven LlamaIndex Workflow

We also include public example documents, so anyone can run this end-to-end.


In [ ]:
# Install and imports
import os
from typing import List, Optional
from datetime import date
from decimal import Decimal
from pydantic import BaseModel, Field

from dotenv import load_dotenv

load_dotenv()

# LlamaIndex workflow imports
from llama_index.core.workflow import (
    Event,
    StartEvent,
    StopEvent,
    Context,
    Workflow,
    step,
)
from llama_index.core.prompts import ChatPromptTemplate
from llama_index.llms.openai import OpenAI

# LlamaCloud classify/extract
from llama_cloud.client import AsyncLlamaCloud
from llama_cloud.types import ClassifierRule, ClassifyParsingConfiguration
from llama_cloud_services.beta.classifier.client import ClassifyClient
from llama_cloud_services import LlamaExtract, ExtractionAgent
from llama_cloud import ExtractConfig
from llama_cloud.core.api_error import ApiError

## Sample documents

We will download four public SEC filings (10-K, 10-Q, 8-K, Proxy) into `examples/classify/data/` and run the workflow over them.


In [ ]:
# Download Microsoft PDFs for all four types
import pathlib
import requests

DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

MSFT_DOCS = {
    "10-K": "https://microsoft.gcs-web.com/static-files/1c864583-06f7-40cc-a94d-d11400c83cc8",
    "10-Q": "https://microsoft.gcs-web.com/static-files/f96f7d38-36ce-4a26-9e29-61701cdca7a7",
    "8-K": "https://microsoft.gcs-web.com/static-files/dc50633a-2880-4303-bebb-bdca89149f65",
    "Proxy": "https://microsoft.gcs-web.com/static-files/d5ec87b3-e29d-4d33-9d84-5ce1f194dcf1",
}

local_files = {}
for k, url in MSFT_DOCS.items():
    out_path = DATA_DIR / f"msft_{k.replace('-', '').lower()}.pdf"
    if not out_path.exists():
        # special case for proxy, run wget
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(out_path, "wb") as f:
            f.write(r.content)
    local_files[k] = str(out_path)

local_files

{'10-K': 'data/msft_10k.pdf',
 '10-Q': 'data/msft_10q.pdf',
 '8-K': 'data/msft_8k.pdf',
 'Proxy': 'data/msft_proxy.pdf'}

## Define type-specific extraction schemas

We define concise Pydantic schemas for 10-K, 10-Q, 8-K, and Proxy (DEF 14A).


In [ ]:
class Form10K(BaseModel):
    company_name: str
    fiscal_year_end: date
    annual_revenue: Optional[Decimal] = None
    net_income: Optional[Decimal] = None
    total_assets: Optional[Decimal] = None
    employee_count: Optional[int] = None
    business_description: str
    primary_risk_factors: List[str]
    business_segments: List[str]
    geographic_markets: List[str]


class Form10Q(BaseModel):
    company_name: str
    quarter_end: date
    quarterly_revenue: Optional[Decimal] = None
    quarterly_net_income: Optional[Decimal] = None
    revenue_change_pct: Optional[float] = None
    material_changes: List[str]
    subsequent_events: List[str]


class Form8K(BaseModel):
    company_name: str
    filing_date: date
    event_date: date
    event_type: str
    material_event_description: str
    financial_impact: Optional[Decimal] = None
    involved_parties: List[str]


class ProxyStatement(BaseModel):
    company_name: str
    meeting_date: date
    ceo_name: str
    ceo_total_compensation: Optional[Decimal] = None
    board_members: List[str]
    executive_officers: List[str]
    shareholder_proposals: List[str]
    voting_matters: List[str]
    audit_firm: Optional[str] = None

## Classification rules

We define four `ClassifierRule` entries describing each SEC form in natural language. The classifier returns the `type` string for the best-matching rule.


In [ ]:
# NOTE: the types need to be in lowercase
SEC_CLASSIFICATION_RULES: list[ClassifierRule] = [
    ClassifierRule(
        type="10-k",
        description=(
            "Annual report on Form 10-K, includes business overview, risk factors, management's"
            " discussion and analysis, audited financial statements for the fiscal year."
        ),
    ),
    ClassifierRule(
        type="10-q",
        description=(
            "Quarterly report on Form 10-Q, includes unaudited quarterly financial statements,"
            " MD&A for the quarter, and updates on risk factors."
        ),
    ),
    ClassifierRule(
        type="8-k",
        description=(
            "Current report on Form 8-K, discloses material events such as acquisitions,"
            " executive changes, earnings releases, or other significant occurrences."
        ),
    ),
    ClassifierRule(
        type="proxy",
        description=(
            "DEF 14A proxy statement for shareholder meetings including proposals and voting,"
            " board of directors, executive compensation (CD&A), and auditor information."
        ),
    ),
]

## Initialize clients

We create clients for classification and extraction. Set `LLAMA_CLOUD_API_KEY` in your environment. Optionally set `LLAMA_CLOUD_BASE_URL`, `LLAMA_CLOUD_PROJECT_ID`, `LLAMA_CLOUD_ORGANIZATION_ID`.


In [ ]:
api_key = os.getenv("LLAMA_CLOUD_API_KEY")
base_url = os.getenv("LLAMA_CLOUD_BASE_URL")
project_id = os.getenv("LLAMA_CLOUD_PROJECT_ID")
organization_id = os.getenv("LLAMA_CLOUD_ORGANIZATION_ID")

if not api_key:
    raise ValueError("LLAMA_CLOUD_API_KEY not set. Please set it in your environment.")

async_client = AsyncLlamaCloud(token=api_key, base_url=base_url)
classify_client = ClassifyClient(
    async_client, project_id=project_id, organization_id=organization_id
)

extract_config = ExtractConfig(extraction_mode="BALANCED")
llama_extract = LlamaExtract(project_id=project_id, organization_id=organization_id)

# Model for LLM summarization in prompts if needed
llm = OpenAI(model="gpt-4o")

## Using Classify Module

In this section we show you how to use the `ClassifyClient` module in a standalone manner (before using it in an e2e workflow).

We run the classification module over the Microsoft 10-K file to verify that the output is in the correct class.


In [ ]:
# set parsing configuration
parsing_config = ClassifyParsingConfiguration(max_pages=5)

# classify file
results = await classify_client.aclassify_file_path(
    rules=SEC_CLASSIFICATION_RULES,
    file_input_path="data/msft_10k.pdf",
    parsing_configuration=parsing_config,
)

The result will not only contain the classification type, but also the reasoning.

In [ ]:
print(results.items[0].result.type)
print(results.items[0].result.reasoning)

10-k
The document is titled 'FORM 10-K' and is labeled as an 'ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934' for the fiscal year ended June 30, 2024. It contains all the hallmarks of a 10-K filing, including a business overview, risk factors, management's discussion and analysis (MD&A), and references to audited financial statements. The index lists all required sections for a 10-K, such as Business, Risk Factors, MD&A, Financial Statements, and more. There is no indication that this is a quarterly report (10-Q), a current report (8-K), or a proxy statement (DEF 14A). The content and structure perfectly match the definition of a 10-K.


## Workflow: Classify then Extract

We build a `Workflow` with steps:
- `classify_file`: upload and classify the document
- `extract_by_type`: create/select an agent for the type and extract the corresponding schema
- `format_output`: return unified JSON with `type` and `data`


In [ ]:
class ClassifiedEvent(Event):
    file_path: str
    doc_type: str


class ExtractedEvent(Event):
    file_path: str
    doc_type: str
    data: dict


def _schema_for_type(doc_type: str):
    if doc_type == "10-k":
        return Form10K
    if doc_type == "10-q":
        return Form10Q
    if doc_type == "8-k":
        return Form8K
    if doc_type == "proxy":
        return ProxyStatement
    raise ValueError(f"Unsupported doc_type: {doc_type}")


def _agent_name_for_type(doc_type: str) -> str:
    return f"sec-{doc_type.lower()}-extractor"


class SECClassifyExtractWorkflow(Workflow):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.agent_registry: dict[str, ExtractionAgent] = {}

    @step
    async def classify_file(self, ctx: Context, ev: StartEvent) -> ClassifiedEvent:
        file_path = ev.file_path
        parsing_config = ClassifyParsingConfiguration(max_pages=5)
        results = await classify_client.aclassify_file_path(
            rules=SEC_CLASSIFICATION_RULES,
            file_input_path=file_path,
            parsing_configuration=parsing_config,
        )
        item = results.items[0]
        doc_type = item.result.type
        return ClassifiedEvent(file_path=file_path, doc_type=doc_type)

    @step
    async def extract_by_type(
        self, ctx: Context, ev: ClassifiedEvent
    ) -> ExtractedEvent:
        schema = _schema_for_type(ev.doc_type)
        agent_name = _agent_name_for_type(ev.doc_type)

        # Lazily create agent if not present
        if ev.doc_type not in self.agent_registry:
            try:
                existing = llama_extract.get_agent(name=agent_name)
                if existing:
                    llama_extract.delete_agent(existing.id)
            except ApiError as e:
                if e.status_code != 404:
                    raise
            agent = llama_extract.create_agent(
                name=agent_name, data_schema=schema, config=extract_config
            )
            self.agent_registry[ev.doc_type] = agent

        extraction = await self.agent_registry[ev.doc_type].aextract(ev.file_path)
        data = (
            extraction.data
            if isinstance(extraction.data, dict)
            else extraction.model_dump()
        )
        return ExtractedEvent(file_path=ev.file_path, doc_type=ev.doc_type, data=data)

    @step
    async def format_output(self, ctx: Context, ev: ExtractedEvent) -> StopEvent:
        return StopEvent(
            result={"type": ev.doc_type, "data": ev.data, "file": ev.file_path}
        )

In [ ]:
# Run the workflow on each Microsoft document
import nest_asyncio

nest_asyncio.apply()

workflow = SECClassifyExtractWorkflow(verbose=True, timeout=None)

results = {}
for doc_type, path in local_files.items():
    print(f"Running workflow for {path}...")
    res = await workflow.run(file_path=path)
    results[doc_type] = res

{t: {"type": v["type"], "file": v["file"]} for t, v in results.items()}

Running workflow for 10-K...
Running step classify_file
Step classify_file produced event ClassifiedEvent
Running step extract_by_type


Extracting files: 100%|██████████| 1/1 [00:19<00:00, 19.01s/it]


Step extract_by_type produced event ExtractedEvent
Running step format_output
Step format_output produced event StopEvent
Running workflow for 10-Q...
Running step classify_file
Step classify_file produced event ClassifiedEvent
Running step extract_by_type


Extracting files: 100%|██████████| 1/1 [00:25<00:00, 25.95s/it]


Step extract_by_type produced event ExtractedEvent
Running step format_output
Step format_output produced event StopEvent
Running workflow for 8-K...
Running step classify_file
Step classify_file produced event ClassifiedEvent
Running step extract_by_type


Extracting files: 100%|██████████| 1/1 [01:58<00:00, 118.02s/it]


Step extract_by_type produced event ExtractedEvent
Running step format_output
Step format_output produced event StopEvent
Running workflow for Proxy...
Running step classify_file
Step classify_file produced event ClassifiedEvent
Running step extract_by_type


Extracting files: 100%|██████████| 1/1 [02:07<00:00, 127.58s/it]

Step extract_by_type produced event ExtractedEvent
Running step format_output
Step format_output produced event StopEvent


{'10-K': {'type': '10-k', 'file': 'data/msft_10k.pdf'},
 '10-Q': {'type': '10-q', 'file': 'data/msft_10q.pdf'},
 '8-K': {'type': '8-k', 'file': 'data/msft_8k.pdf'},
 'Proxy': {'type': 'proxy', 'file': 'data/msft_proxy.pdf'}}

In [ ]:
# Pretty print a subset of fields for each type
import json
from pprint import pprint


def summarize(doc_type: str, data: dict):
    print(f"\n==== {doc_type} ====")
    if doc_type == "10-k":
        keys = [
            "company_name",
            "fiscal_year_end",
            "annual_revenue",
            "net_income",
            "primary_risk_factors",
        ]
    elif doc_type == "10-q":
        keys = [
            "company_name",
            "quarter_end",
            "quarterly_revenue",
            "revenue_change_pct",
            "material_changes",
        ]
    elif doc_type == "8-k":
        keys = [
            "company_name",
            "event_date",
            "event_type",
            "material_event_description",
        ]
    else:  # Proxy
        keys = [
            "company_name",
            "meeting_date",
            "ceo_name",
            "ceo_total_compensation",
            "voting_matters",
        ]
    subset = {k: data.get(k) for k in keys}
    pprint(subset)


for t, out in results.items():
    summarize(out["type"], out["data"])


==== 10-k ====
{'annual_revenue': 245122000000.0,
 'company_name': 'Microsoft Corporation',
 'fiscal_year_end': 'June 30, 2024',
 'net_income': 88136000000.0,
 'primary_risk_factors': ['We face intense competition across all markets for '
                          'our products and services from a range of '
                          'competitors varying in size and specialization. '
                          'Barriers to entry are low in many markets, and we '
                          'experience rapid evolution in technologies and user '
                          'needs. Competition includes firms with competing '
                          'platforms and business models such as cloud-based '
                          'services and open source software. We are investing '
                          'in AI as a highly competitive area. Cybersecurity '
                          'threats from nation-state actors and other '
                          'malicious parties pose significant r